# 04: LRCN + Data Augmentation
**Group 11 · COSE474 Deep Learning · Korea University · Spring 2026**

## Overview

This notebook trains **LRCN + Data Augmentation** — the same CNN+LSTM architecture from notebook 03 (Marcello), with augmentation added to the training pipeline.

**Purpose in the ablation study:**
Isolate the contribution of data augmentation. Architecture, optimizer, LR, and val transforms are **identical to notebook 03** — only the training transforms differ. Any accuracy change is purely from augmentation.

**Why augmentation for KSL?**
KSL-77 has only ~16 training videos per class. Augmentation artificially expands the effective training set by creating varied versions of existing clips — different lighting, angles, and signing speeds.

**Augmentation applied:**
- Spatial (per frame): random flip, rotation ±15°, brightness/contrast jitter, random resized crop
- Temporal (per clip): frame speed variation (0.8x–1.2x) simulating different signing speeds

**Critical:** Val transforms are **clean** (identical to notebook 03). Augmentation is training-only.

**Settings kept identical to notebook 03:**
- LRCN architecture: exact same class (copied from Marcello)
- Batch size: 8 (memory constraint — 16 frames per clip on T4)
- Learning rate: 1e-4
- Optimizer: Adam, Scheduler: StepLR (step=10, gamma=0.5)
- Signer split: 00-15 train, 16-19 val

**Compare against:** Notebook 03 LRCN baseline = **8.53%**

**Outputs:**
```
models/checkpoints/lrcn_augmented.pth
results/figures/04_augmentation_curves.png
results/figures/04_augmentation_confusion.png
results/logs/04_augmentation_log.csv
```

# Mount Drive + config

In [2]:
# Author: Hakeemi
# Notebook 04 — LRCN + Data Augmentation
# LRCN architecture: copied exactly from 03_CNN_LSTM.ipynb (Marcello)

from google.colab import drive
drive.mount('/content/drive')

import os, sys
sys.path.append('/content/drive/MyDrive/KSL_DL2026')
import config

# Use local frames if available — faster I/O than Drive (same as notebook 03)
LOCAL_FRAMES_DIR = '/content/frames/'
if os.path.exists(LOCAL_FRAMES_DIR):
    print(f"Found local frames at {LOCAL_FRAMES_DIR}. Overriding Drive path.")
    config.DATA_FRAMES = LOCAL_FRAMES_DIR
else:
    print(f"Local frames not found. Using Drive: {config.DATA_FRAMES}")

print(f"Classes    : {config.NUM_CLASSES}")
print(f"Frames/clip: {config.NUM_FRAMES}")

Mounted at /content/drive
Local frames not found. Using Drive: /content/drive/MyDrive/KSL_DL2026/data/frames
Classes    : 77
Frames/clip: 16


# Imports

In [3]:
import csv, random, datetime
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

Device : cpu


# Dataset

`KSLDataset` — identical to notebook 01 and 03. Returns `(NUM_FRAMES, C, H, W)` per clip.
Used for **val set only** — no augmentation on val.

`KSLAugDataset` — extends KSLDataset with temporal augmentation (frame speed variation).
Used for **train set only**.

In [4]:
class KSLDataset(Dataset):
    """Identical to 01_data_pipeline and 03_CNN_LSTM. Used for val set."""
    def __init__(self, clip_dirs, transform=None):
        self.clips     = clip_dirs
        self.transform = transform

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path   = self.clips[idx]
        folder_name = os.path.basename(clip_path)
        class_id    = int(folder_name.split('_')[1])
        label       = class_id - 1

        frame_files = sorted(os.listdir(clip_path))
        if len(frame_files) < config.NUM_FRAMES:
            frame_files += [frame_files[-1]] * (config.NUM_FRAMES - len(frame_files))
        frame_files = frame_files[:config.NUM_FRAMES]

        tensors = []
        for fname in frame_files:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)

        return torch.stack(tensors), label


class KSLAugDataset(Dataset):
    """
    KSLDataset + temporal augmentation (random frame speed variation).
    Spatial augmentation is handled by the train_transform.
    Used for TRAINING only.

    Temporal augmentation:
        speed_factor ~ Uniform(AUG_SPEED_MIN, AUG_SPEED_MAX)
        < 1.0 = slower signer (wider sampling window)
        > 1.0 = faster signer (narrower sampling window)
    """
    def __init__(self, clip_dirs, transform=None,
                 speed_min=config.AUG_SPEED_MIN,
                 speed_max=config.AUG_SPEED_MAX):
        self.clips     = clip_dirs
        self.transform = transform
        self.speed_min = speed_min
        self.speed_max = speed_max

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path    = self.clips[idx]
        folder_name  = os.path.basename(clip_path)
        class_id     = int(folder_name.split('_')[1])
        label        = class_id - 1
        frame_files  = sorted(os.listdir(clip_path))
        total_frames = len(frame_files)

        # temporal augmentation — random sampling window
        speed_factor = random.uniform(self.speed_min, self.speed_max)
        window       = int(config.NUM_FRAMES * speed_factor)
        window       = max(window, config.NUM_FRAMES)
        window       = min(window, total_frames)
        start        = max(0, (total_frames - window) // 2)
        end          = min(total_frames - 1, start + window)
        indices      = np.linspace(start, end, config.NUM_FRAMES, dtype=int)
        selected     = [frame_files[min(i, total_frames - 1)] for i in indices]

        tensors = []
        for fname in selected:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)

        return torch.stack(tensors), label

# Signer-based Train/Val Split
Identical to notebook 03 — signers 00-15 train, 16-19 val.

In [6]:
TRAIN_SIGNERS = {f"{i:02d}" for i in range(16)}
VAL_SIGNERS   = {f"{i:02d}" for i in range(16, 20)}

all_clips = sorted([
    os.path.join(config.DATA_FRAMES, d)
    for d in os.listdir(config.DATA_FRAMES)
    if os.path.isdir(os.path.join(config.DATA_FRAMES, d))
])

train_clips = [c for c in all_clips if os.path.basename(c).split('_')[0] in TRAIN_SIGNERS]
val_clips   = [c for c in all_clips if os.path.basename(c).split('_')[0] in VAL_SIGNERS]

print(f"Total clips : {len(all_clips)}")
print(f"Train clips : {len(train_clips)}  (signers 00-15)")
print(f"Val clips   : {len(val_clips)}   (signers 16-19)")

Total clips : 1229
Train clips : 971  (signers 00-15)
Val clips   : 258   (signers 16-19)


# Transforms

Train: WITH spatial augmentation.
Val: identical to notebook 03 — NO augmentation.

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=config.AUG_ROTATION),
    transforms.ColorJitter(brightness=config.AUG_BRIGHTNESS,
                           contrast=config.AUG_CONTRAST),
    transforms.RandomResizedCrop(size=config.IMG_SIZE,
                                 scale=(config.AUG_CROP, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.NORMALIZE_MEAN, std=config.NORMALIZE_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.NORMALIZE_MEAN, std=config.NORMALIZE_STD)
])

BATCH = 8  # same as notebook 03 — memory constraint for 16-frame clips on T4

train_dataset = KSLAugDataset(train_clips, train_transform)
val_dataset   = KSLDataset(val_clips, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Batch size    : {BATCH}")
frames, lbl = train_dataset[0]
print(f"Clip shape    : {frames.shape}")  # (16, 3, 224, 224)

Train samples : 971
Val samples   : 258
Batch size    : 8
Clip shape    : torch.Size([16, 3, 224, 224])


# Model — LRCN

**Exact copy of Marcello's `LRCN` class from `03_CNN_LSTM.ipynb`.**
Architecture unchanged — only training data differs. Essential for valid ablation.

Key design decisions in forward pass:
1. Reshape `(B,T,C,H,W) → (B*T,C,H,W)` — all frames through VGG16 in one call (fast)
2. `torch.no_grad()` on frozen backbone — saves memory
3. `Linear(25088→512)` + BatchNorm + ReLU + Dropout — reduces dim before LSTM
4. LSTM reads `(B, T, 512)` → final hidden state `h_T` summarizes the full clip
5. `Linear(256→77)` → class logits

In [8]:
class LRCN(nn.Module):
    """
    VGG16 (frozen, ImageNet) -> per-frame feature -> LSTM -> classifier.
    Copied exactly from 03_CNN_LSTM.ipynb (Marcello Nico Valerian).
    """
    def __init__(self,
                 num_classes  = config.NUM_CLASSES,
                 feature_dim  = 512,
                 lstm_hidden  = config.LSTM_HIDDEN,
                 lstm_layers  = config.LSTM_LAYERS,
                 bidirectional= config.LSTM_BIDIRECT):
        super(LRCN, self).__init__()

        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.avgpool  = vgg.avgpool

        for p in self.features.parameters():
            p.requires_grad = False

        self.frame_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.lstm = nn.LSTM(
            input_size    = feature_dim,
            hidden_size   = lstm_hidden,
            num_layers    = lstm_layers,
            batch_first   = True,
            bidirectional = bidirectional,
        )

        out_dim = lstm_hidden * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(out_dim, num_classes),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        with torch.no_grad():
            x = self.features(x)
            x = self.avgpool(x)
        x = self.frame_fc(x)
        x = x.view(B, T, -1)
        out, (h_n, _) = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)


model = LRCN().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {total - trainable:,}")
print(f"Total params     : {total:,}")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 150MB/s]


Trainable params : 13,654,861
Frozen params    : 14,714,688
Total params     : 28,369,549


# Training Setup
Identical to notebook 03. lr=1e-4 (not config default) — more stable for LSTM training.

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=config.WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f"Loss      : CrossEntropyLoss")
print(f"Optimizer : Adam (lr=1e-4, wd={config.WEIGHT_DECAY})")
print(f"Scheduler : StepLR (step=10, gamma=0.5)")
print(f"Epochs    : {config.NUM_EPOCHS}, Patience : {config.PATIENCE}")

Loss      : CrossEntropyLoss
Optimizer : Adam (lr=1e-4, wd=0.0001)
Scheduler : StepLR (step=10, gamma=0.5)
Epochs    : 50, Patience : 10


# Train and Validation Functions

In [10]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(clips)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += logits.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            logits = model(clips)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds       = logits.argmax(1)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100.0 * correct / total, all_preds, all_labels

## Run Training Loop
**Direct comparison: notebook 03 LRCN baseline = 8.53%**
If this > 8.53% → augmentation helps. If similar → augmentation has limited effect (also valid finding).

In [ ]:
history = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc']}
best_val_acc, patience_count = 0.0, 0
best_preds, best_labels      = [], []

print("Starting training (LRCN + Augmentation)...\n")
print(f"{'Epoch':>5} {'Tr Loss':>9} {'Tr Acc':>8} {'Vl Loss':>9} {'Vl Acc':>8}")
print('─' * 48)

for epoch in range(1, config.NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc, vl_preds, vl_labels = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    for k, v in zip(['train_loss','train_acc','val_loss','val_acc'],
                    [tr_loss, tr_acc, vl_loss, vl_acc]):
        history[k].append(v)

    marker = ' <-- best' if vl_acc > best_val_acc else ''
    print(f"{epoch:>5} {tr_loss:>9.4f} {tr_acc:>7.2f}% {vl_loss:>9.4f} {vl_acc:>7.2f}%{marker}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc; best_preds = vl_preds
        best_labels  = vl_labels; patience_count = 0
        torch.save(model.state_dict(), config.CKPT_AUG)
    else:
        patience_count += 1

    if patience_count >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}"); break

print(f"\nBest val accuracy : {best_val_acc:.2f}%")
print(f"LRCN baseline     : 8.53%")
print(f"Delta from aug    : {best_val_acc - 8.53:+.2f} pp")
print(f"Checkpoint        : {config.CKPT_AUG}")

Starting training (LRCN + Augmentation)...

Epoch   Tr Loss   Tr Acc   Vl Loss   Vl Acc
────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


# Results
## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('LRCN + Augmentation — Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('LRCN + Augmentation — Accuracy'); ax2.legend()
plt.tight_layout()
path = f"{config.RESULTS_FIGS}/04_augmentation_curves.png"
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

## Confusion matrix
Compare against `03_lrcn_confusion.png`. If augmentation helped, class pairs confused in notebook 03 should improve here.

In [ ]:
cm = confusion_matrix(best_labels, best_preds)
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=range(1, 78), yticklabels=range(1, 78))
plt.title(f'LRCN + Augmentation — Confusion Matrix  (Val Acc: {best_val_acc:.2f}%)', fontsize=13)
plt.ylabel('True class'); plt.xlabel('Predicted class')
plt.tight_layout()
path = f"{config.RESULTS_FIGS}/04_augmentation_confusion.png"
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

# Log Results
Update 고동우's `04_Augmentation` sheet in `KSL_experiments.xlsx` with `best_val_acc`.

In [ ]:
row = {
    'experiment':        '04_lrcn_augmentation',
    'date':              datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model':             'LRCN (VGG16 frozen + LSTM) + Augmentation',
    'temporal_modeling': True,
    'augmentation':      True,
    'transfer_learning': False,
    'aug_flip':          config.AUG_FLIP,
    'aug_rotation':      config.AUG_ROTATION,
    'aug_brightness':    config.AUG_BRIGHTNESS,
    'aug_speed_min':     config.AUG_SPEED_MIN,
    'aug_speed_max':     config.AUG_SPEED_MAX,
    'train_signers':     '00-15',
    'val_signers':       '16-19',
    'lstm_hidden':       config.LSTM_HIDDEN,
    'batch_size':        BATCH,
    'learning_rate':     1e-4,
    'best_val_acc':      round(best_val_acc, 2),
    'lrcn_baseline_acc': 8.53,
    'delta_from_lrcn':   round(best_val_acc - 8.53, 2),
    'epochs_run':        len(history['val_acc']),
    'notes':             'LRCN + spatial + temporal aug. Compare vs 03 (8.53%).'
}
log_path = f"{config.RESULTS_LOGS}/04_augmentation_log.csv"
write_header = not os.path.exists(log_path)
with open(log_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header: writer.writeheader()
    writer.writerow(row)
print("Results logged:")
for k, v in row.items(): print(f"  {k}: {v}")
print(f"\nSaved to: {log_path}")

# Summary
1. Update `04_Augmentation` sheet in `KSL_experiments.xlsx`
2. Update `Summary` ablation table
3. Upload to GitHub

In [ ]:
print('=' * 55)
print('  LRCN + AUGMENTATION COMPLETE')
print('=' * 55)
print(f"  Best val accuracy  : {best_val_acc:.2f}%")
print(f"  LRCN baseline (03) : 8.53%")
print(f"  Delta              : {best_val_acc - 8.53:+.2f} pp")
print()
print("  Ablation table so far:")
print(f"  Simple CNN          :  6.98%  (notebook 02)")
print(f"  LRCN baseline       :  8.53%  (notebook 03)")
print(f"  LRCN + aug (this)   :  {best_val_acc:.2f}%  (notebook 04)")
print(f"  LRCN + transfer     :  TBD    (notebook 05 - Marcello)")
print(f"  LRCN + aug + xfer   :  TBD    (notebook 06 - Hakeemi)")
print()
print(f"  Checkpoint : {config.CKPT_AUG}")
print(f"  Curves     : results/figures/04_augmentation_curves.png")
print(f"  Confusion  : results/figures/04_augmentation_confusion.png")
print(f"  Log        : results/logs/04_augmentation_log.csv")
print()
print("  --> Next: 05_transfer_learning.ipynb (Marcello)")
print('=' * 55)